# Geographical Analysis of Touristic Rentals in Catalonia 

* **Author:** Lídia Vicente Sánchez
* **Project:** `Speculation-Map` — Big Data Analysis and Visualization (University of Barcelona)

### Notebook Objective
This notebook explores the territorial distribution of the number of touristic establishments across Catalan municipalities and comarques. Specifically, it analyzes:
1. **Touristic rentals distributed by municipalities and by comarques:** Studies of the territorial distribution of tourist establishments, focusing on 2025. (static map)
2. **Touristic rentals per inhabitants distributed by municipalities and by comarques:** Studies of the territorial distribution of tourist establishments per inhabitant, focusing on 2025. (static and interactive maps)
3. **Relation between rent price and number of touristic establishments:** Creation of a plot relating rent prices and touristic rentals per municipality. 

### AI Usage:

No AI was used in the development of the code.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd



# Creation of the touristic rentals by municipality map
## Our dataset 
It presents the following columns:

['primer_cognom', 'nom_del_titular', 'ra_social_del_titular', 'cif',
       'marca_tur_stica', 'sstt', 'total_estances', 'total_places',
       'unitat_d_allotjament', 'categoria', 'modalitat', 'grup', 'prov_ncia',
       'codi_comarca_idescat', 'comarca', 'codi_municipi_idescat', 'municipi',
       'codi_postal', 'estat', 'r_tol', 'd_git_de_control', 'n_mero_inscripci',
       'tipus_establiment', 'cedula_d_habitabilitat', 'referencia_cadastral',
       'bloc', 'escala', 'porta', 'pis', 'numero', 'nom_de_la_via',
       'tipus_de_via']

In [ ]:
touristicrentals = pd.read_csv(r"../../Speculation-Map/Data/TouristicRentals/prova2025.csv",
                       index_col=0,dtype=object)
touristicrentals.head()

## Dataframe named count counts the number of touristic rentals for each municipality by counting the number of rows with the same "codi_municipi_idescat" number

In [ ]:
count = (
    touristicrentals["codi_municipi_idescat"]
    .value_counts()
    .reset_index()
)
count.columns = ["codi_municipi_idescat", "TouristicRentals"]
count.head()

We change the name of codi_municipi_idescat to CODIMUNI.

In [ ]:
count.rename(columns={"codi_municipi_idescat": "CODIMUNI"},inplace=True)

## Municipalities dataset is used for the creation of the Catalonia map distributed by municipalities
It presents the following columns:
['CODIMUNI', 'NOMMUNI', 'CAPMUNI', 'AREAM5000', 'CODICOMAR', 'NOMCOMAR',
       'CAPCOMAR', 'CODIVEGUE', 'NOMVEGUE', 'CAPVEGUE', 'CODIPROV', 'NOMPROV',
       'CAPPROV', 'geometry']

In [ ]:
municipalities=gpd.read_file("../../Speculation-Map/Data/CatalunyaMapMunicipalities.json").to_crs(25831)
municipalities.head()

## Creation of our main dataframe 'df' by merging TouristicRentals and Municipalities dataset

In [ ]:
df = municipalities.merge(count, on="CODIMUNI", how="left", validate="one_to_one")

## Distribution of Touristic Rentals by municipality

In [ ]:
df.plot(column="TouristicRentals", cmap="Purples", legend=True)

This map is not representative as municipalities such as Barcelona, which are very crowded, present a larger number of touristic establishments. Noticing this, a touristic rentals/number of inhabitants map is created.

# Creation of the touristic rentals/number of inhabitants maps
### Inhabitants dataset 
It presents the following columns [YEAR', 'MUN', 'SEX', 'CONCEPT', 'status', 'value'].

The number of inhabitants is shown in three rows for each municipality, first one for males, second one for females, third one for the sum.

In [ ]:
inhabitants = gpd.read_file("../../Speculation-Map/Data/Habitants.csv")
inhabitants.head()


### Inhabitants_pivot dataframe uses inhabitants dataset information and organises the data 
It presents the following columns: ['F': Number of females, 'M': Number of males, 'TOTAL': Total number of inhabitants, 'MUN_COPY': Copy of the municipality code, as it presented as an index and we will need it as a column.],

In [ ]:
inhabitants_pivot = inhabitants.pivot(index="MUN",columns="SEX",values="value")
inhabitants_pivot["MUN_COPY"] = inhabitants_pivot.index
inhabitants_pivot.head()

Again, we change the name of codi_municipi_idescat to CODIMUNI.

In [ ]:
inhabitants_pivot.rename(columns={"MUN_COPY": "CODIMUNI"},inplace=True)

### Inhabitants_pivot is merged with our main dataframe.

In [ ]:
df = df.merge(inhabitants_pivot, on="CODIMUNI", how="left", validate="one_to_one")


"TOTAL" is the only column we are interested in. For further operations, we need it in float type and we rename it "Inhab".


In [ ]:
df["Inhab"]=pd.to_numeric(df["TOTAL"], errors="coerce") 


### Creation of a new column "fraction"
It presents the fraction of number of touristic rentals per number of inhabitants.

In [ ]:
df["fraction"] = df["TouristicRentals"]/df["Inhab"]

## Grouping by comarca
### First, we add to the dataframe a column with the sum of inhabitants for each comarca, called 'Inhab_com'
Different municipalities in the same comarca will share the same value in this column.

In [ ]:
total_hab_com=df.groupby("CODICOMAR",as_index=False).agg(Inhab_com=("Inhab",lambda s: s.sum()))
df=df.merge(total_hab_com,on="CODICOMAR",how="left",validate="many_to_one")
df["Inhab_com"]=pd.to_numeric(df["Inhab_com"], errors="coerce") 

### We repeat the process with the variable TouristicRentals, in a new column called 'TouristicRentals_com'.

In [ ]:
total_tour_com=df.groupby("CODICOMAR",as_index=False).agg(TouristicRentals_com=("TouristicRentals",lambda s: s.sum()))
df=df.merge(total_tour_com,on="CODICOMAR",how="left",validate="many_to_one")

### Dividing this two new columns, we get a new column 'fraction_com'.
It presents the fraction of number of touristic rentals per number of inhabitants, with the information grouped by comarques.

In [ ]:
df["fraction_com"]=df["TouristicRentals_com"]/df["Inhab_com"]

### At this point, our main dataframe contains all the information needed

In [ ]:
df.head(10)

# Distribution of Touristic Rentals/Number of inhabitants by municipality and by comarca

### Static plots (logarithmic scale)

In [ ]:
from matplotlib.colors import LogNorm
fig, axs = plt.subplots(1, 2, figsize=(11, 5),  layout="constrained")
df.plot(column="fraction", cmap="Purples", norm=LogNorm(),legend=True, ax=axs[0],legend_kwds={"shrink": 0.7})
df.plot(column="fraction_com",cmap="Purples",norm=LogNorm(),legend=True,ax=axs[1],legend_kwds={"shrink": 0.7})
axs[0].set_title("Touristic Rentals/inhabitants by municipalities")
axs[1].set_title("Touristic Rentals/inhabitants by comarques")

for ax in axs:
    ax.set_axis_off()
#add_scale_bar(axs)
#add_north_arrow(axs)
plt.show()

### Static plots by quartiles

In [ ]:
df["quartiles"] = pd.qcut(df["fraction"], 4, labels=["lowest 20%", "2", "3", "highest 20%"])
df["quartiles_com"] = pd.qcut(df["fraction_com"], 4, labels=["lowest 20%", "2", "3", "highest 20%"])

In [ ]:
from matplotlib.colors import LogNorm 
fig, axs = plt.subplots(1, 2, figsize=(11, 5),  layout="constrained")
df.plot(column="quartiles", cmap="Purples", norm=LogNorm(),legend=True, ax=axs[0])
df.plot(column="quartiles_com",cmap="Purples",norm=LogNorm(),legend=True,ax=axs[1])
axs[0].set_title("Touristic Rentals/inhabitants by municipalities")
axs[1].set_title("Touristic Rentals/inhabitants by comarques")

for ax in axs:
    ax.set_axis_off()
plt.show()

### Interactive plots
LogNorm function is not available in interactive plots. We create a new column for the logarithm of "fraction". 
#### By municipalities


In [ ]:
df["log"]=np.log(df["fraction"])
df["logquartiles"] = pd.qcut(df["log"], 4, labels=["1", "2", "3", "4"])
df["logquartiles"]=pd.to_numeric(df["logquartiles"], errors="coerce")
df["TouristicRentals/10000Inhabitants"]=df["fraction"]*10000



In [ ]:
import plotly.express as px

municipis=gpd.read_file("../../Speculation-Map/Data/CatalunyaMapMunicipalities.json")

df["geometry"]=municipis["geometry"]

fig = px.choropleth_map(
    df, geojson=df.__geo_interface__, locations="CODIMUNI",
    featureidkey="properties.CODIMUNI", color="logquartiles", color_continuous_scale="Purples",
    hover_data=["NOMMUNI","NOMCOMAR","TouristicRentals/10000Inhabitants"],
    center={"lat": 41.7, "lon": 1.7}, zoom=6.3, map_style="carto-positron",
    labels={"logquartiles": "rank"},
)

fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig

#### By comarques
Creation of new dataframe with the data in the main one, grouped by comarques. All the '_com' variables were repeated in the main df, so we can take the first value. We use the mean in 'rank' so we don't have to convert it to numeric.

In [ ]:
df["log_com"]=np.log(df["fraction_com"])
df["logquartiles_com"]=pd.qcut(df["log_com"],4,labels=["1", "2", "3", "4"])
df["logquartiles_com"]=pd.to_numeric(df["logquartiles_com"],errors="coerce")
com=df.groupby("CODICOMAR",as_index=False).agg(rank=("logquartiles_com",lambda s: s.mean()), NOMCOMAR=("NOMCOMAR", "first"),fraction_com=("fraction_com","first"),Inhab_com=("Inhab_com","first"))
com["TouristicRentals/10000Inhabitants"]=com["fraction_com"]*10000
com.head()

In [ ]:
import plotly.express as px

comarques=gpd.read_file("../../Speculation-Map/Data/CatalunyaMapComarques.json")



fig = px.choropleth_map(
    com, geojson=comarques.__geo_interface__, locations="CODICOMAR",
    featureidkey="properties.CODICOMAR", color="rank", color_continuous_scale="Purples",
    hover_data=["NOMCOMAR","TouristicRentals/10000Inhabitants"],
    center={"lat": 41.7, "lon": 1.7}, zoom=6.3, map_style="carto-positron",
    labels={"Edificis/10000Habitants": "TouristicRentals/10000Inhabitants"},
)

fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig